# Minimal model comparison between versions of DNM-CNM hybrid models and a DNM-DDM hybrid model
**In this comparison I am using a new subjective h estimation**

This notebook follows a simplified version of the modeling framework from Prof. Musslick's cognitive modeling lecture (winter semester 2024).
To improve the validity of the modelling results one should the following best practices
that are not implemented in this branch:
1. Before deciding on a modeling approach one should decide which questions the modeling process is supposed to help answer.
2. Before fitting the models to participant data, one should generate synthetic data from the models based on different given "true" parameters. Then one should check how well these paramaters can be recovered by fitting the models to the synthetic data and comparing the true to the fitted parameters. Then one should fit each model to data generated by the other models and by itself, to validate model recovery.
3. After fitting the models to generated data and before reporting the results one should validate the models by exploring in detail which parts of the experimental data they can and cannot explain.



## Step 0: Setup and imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Resolve repository root from either repo root or src/elias working directories.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
# Handle case where we're in src/elias/... - go up to Glaze root
while REPO_ROOT.name == 'elias' or (REPO_ROOT.parent / 'src' / 'elias').exists():
    if (REPO_ROOT / 'src' / 'elias').exists():
        break
    REPO_ROOT = REPO_ROOT.parent

# Ensure both top-level src and src/elias are importable for notebook execution.
SRC_ROOT = REPO_ROOT / 'src'
ELIAS_SRC = SRC_ROOT / 'elias'
for import_path in (SRC_ROOT, ELIAS_SRC):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from elias_models import fit_models_train_split, score_models_test_split
from elias_models.data_pipeline import (
    attach_subjective_h_from_train,
    build_normative_belief_columns,
    fit_blockwise_subjective_h_choice_only,
    load_participant_data,
    preprocess_loaded_participant_data,
)

DATA_CSV_PATH = REPO_ROOT / 'data' / 'participants.csv'
print(f'REPO_ROOT = {REPO_ROOT}')
print(f'DATA_CSV_PATH = {DATA_CSV_PATH}')

## Step 1: Load and preprocess data

In [ ]:
# Load raw participant rows and normalize choice coding.
df_loaded = load_participant_data(csv_path=str(DATA_CSV_PATH))

# Apply filtering and TRAIN/TEST split assignment using existing policy.
preprocessing_output = preprocess_loaded_participant_data(df_loaded)
df_preprocessed = preprocessing_output['df_all'].copy()

print(f'Rows loaded: {len(df_loaded)}')
print(f'Rows after preprocessing: {len(df_preprocessed)}')
print(f"Rows removed: {preprocessing_output['removed_n']}")

# Show participant/block structure after exclusions.
display(preprocessing_output['participant_structure_table'])

## Step 2: Fit subjective hazard and build normative state

In [ ]:
# Fit one subjective hazard per participant-block from TRAIN choices only.
subjective_h_table = fit_blockwise_subjective_h_choice_only(df_preprocessed, beta=1.0)

# Attach those fitted hazard values to all rows for each participant-block.
df_with_h = attach_subjective_h_from_train(df_preprocessed, subjective_h_table)

# Rebuild prior and posterior normative belief columns used by all models.
df_model = build_normative_belief_columns(df_with_h)

print('Subjective-H fits (participant-block):')
display(subjective_h_table)

print('Modeling frame preview:')
display(df_model[['participant_id', 'block_id', 'trial_index', 'split', 'H', 'prev_normative_belief_L', 'normative_belief_L']].head())

## Step 3: Fit candidate models on TRAIN

### Why Step 3 uses `choice_only` fitting

Step 2 produced the model-ready state variables (`H`, `prev_normative_belief_L`, `normative_belief_L`) for each trial. Step 3 now fits each candidate model's free parameters on TRAIN rows only.

We use `fit_objective='choice_only'` in Step 3 for robustness:
- The TRAIN data per participant-block is limited, so a joint objective can overfit RT noise.
- RT likelihood is estimated from finite simulations, which adds extra Monte Carlo/histogram noise during optimization.
- Choice-only fitting gives a stabler parameter search while still learning the decision policy from TRAIN behavior.

### Which parameters are fit in Step 3

Free (optimized) parameters by model:
- `cont_threshold`: `thr_b1`, `thr_b2`, `thr_b3`, `thr_b4`, `t0`, `g`
- `cont_asymptote`: `asy_b1`, `asy_b2`, `asy_b3`, `asy_b4`, `t0`, `g`
- `ddm_dnm`: `a`, `t0`, `k_v`, `k_z`

Not fit in Step 3:
- `H` is already estimated in Step 2 and then carried forward.
- Values in `fixed_model_params` (e.g., `dt_ms`, `min_duration_ms`, `max_duration_ms`) are fixed during optimization.

In Step 4, we still evaluate fitted parameters using held-out TEST data with both choice and RT (joint score). This checks whether the model that learned robustly from choices also generalizes to reaction-time structure.

Step 5 then summarizes TEST scores and TEST diagnostics for transparent model comparison.



In [ ]:
# Single-pass fit config for all candidate models.
import os
from datetime import datetime
import pickle

FIT_CONFIG = {
    'n_starts': 2,  # number of random initializations for the optimizer
    'n_iterations': 3,  # max optimizer steps per start (kept low for speed)
    'n_sims_per_trial': 100,  # Monte Carlo simulations per trial for likelihood estimation
    'fit_objective': 'choice_only',  # objective to optimize (choices only, no RT term)
    'fixed_model_params': {
        'dt_ms': 1,  # simulation time step in milliseconds
        'min_duration_ms': 150.0,  # minimum simulated decision duration
        'max_duration_ms': 6000.0,  # maximum simulated decision duration
    },
}

AVAILABLE_CPUS = os.cpu_count() or 1
SAFE_MAX_WORKERS = max(1, AVAILABLE_CPUS - 1)
N_JOBS_FIT = max(1, min(14, SAFE_MAX_WORKERS))
print(f'Step 3 workers: {N_JOBS_FIT} (CPUs detected: {AVAILABLE_CPUS})')

fit_output = fit_models_train_split(
    df_model,
    fit_config=FIT_CONFIG,
    random_seed=42,
    n_jobs=N_JOBS_FIT,
)
fit_table = fit_output['fit_table'].copy()

save_fit = False  # Set to True to save fit results for later use in step 4

if save_fit == True:

    # Create output directory if it doesn't exist
    output_dir = REPO_ROOT / 'data'/ 'elias'/ '2026_02_18_run'
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save fit_output dictionary (contains fit_results needed for step 4)
    fit_output_path = output_dir / 'fit_output.pkl'
    with open(fit_output_path, 'wb') as f:
        pickle.dump(fit_output, f)
    
    # Save fit_table as CSV for easy inspection
    fit_table_path = output_dir / 'fit_table.csv'
    fit_table.to_csv(fit_table_path, index=False)
    
    print(f'Saved fit_output to: {fit_output_path}')
    print(f'Saved fit_table to: {fit_table_path}')

print('TRAIN fit summary:')
display(fit_table)


## Step 4: Score fitted models on TEST

In [ ]:
load_fitted_data = True  # Set to True to load previously saved fit results for step 4

if load_fitted_data == True:
    # Load previously saved fit_output from the specified run directory
    output_dir = REPO_ROOT / 'data' / 'elias'/ '2026_02_18_run'
    fit_output_path = output_dir / 'fit_output.pkl'

    with open(fit_output_path, 'rb') as f:
        fit_output = pickle.load(f)

    fit_table = fit_output['fit_table'].copy()

    print(f'Loaded fit_output from: {fit_output_path}')
    print('TRAIN fit summary:')
    display(fit_table)

In [ ]:
# Score fitted model parameters on pooled TEST rows via simulation likelihood.
import os
AVAILABLE_CPUS = os.cpu_count() or 1
SAFE_MAX_WORKERS = max(1, AVAILABLE_CPUS - 1)
N_JOBS_SCORE = max(1, min(14, SAFE_MAX_WORKERS))
print(f'Step 4 workers: {N_JOBS_SCORE} (CPUs detected: {AVAILABLE_CPUS})')

score_output = score_models_test_split(
    df_model,
    fitted_models=fit_output['fit_results'],
    n_sims_per_trial=200,
    rt_bin_width_ms=20.0,
    rt_max_ms=5000.0,
    eps=1e-12,
    random_seed=10_000,
    n_jobs=N_JOBS_SCORE,
)
score_table = score_output['score_table'].copy()

save_score = True

if save_score == True:

    # Create output directory if it doesn't exist
    output_dir = REPO_ROOT / 'data'/  'elias'/ '2026_02_18_run'
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save score_output dictionary (contains trial_scores needed for step 5)
    score_output_path = output_dir / 'score_output.pkl'
    with open(score_output_path, 'wb') as f:
        pickle.dump(score_output, f)

    # Save score_table as CSV for easy inspection
    score_table_path = output_dir / 'score_table.csv'
    score_table.to_csv(score_table_path, index=False)

    print(f'Saved score_output to: {score_output_path}')
    print(f'Saved score_table to: {score_table_path}')

print(f"Held-out winner (lowest TEST joint score): {score_output['winner_model_name']}")
display(score_table)


## Step 5: Summary and Plot

In [ ]:
load_score_output = False  # Set to True to load previously saved score results for step 5
if load_score_output == True:
    # Load previously saved score_output from the specified run directory
    output_dir = REPO_ROOT / 'data' / 'elias' / '2026_02_18_run'
    score_output_path = output_dir / 'score_output.pkl'

    with open(score_output_path, 'rb') as f:
        score_output = pickle.load(f)

    score_table = score_output['score_table'].copy()

    print(f'Loaded score_output from: {score_output_path}')
    print('TEST score summary:')
    display(score_table)


In [ ]:
# Summarize TEST-only model scores and diagnostics.
overview_table = score_table.copy()
overview_table = overview_table.sort_values(['joint_score_test', 'model_name']).reset_index(drop=True)
model_order = overview_table['model_name'].astype(str).tolist()

# Long trial-level table for diagnostic plots.
trial_scores_long = pd.concat(score_output['trial_scores_by_model'].values(), ignore_index=True)
trial_scores_long['model_name'] = trial_scores_long['model_name'].astype(str)

print('Compact TEST-only overview:')
display(overview_table)

# 1) Model ranking by held-out Joint NLL (lower is better).
metric_df = overview_table.set_index('model_name').loc[model_order]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(model_order, metric_df['joint_score_test'], color='steelblue')
ax.set_title('TEST Joint NLL by Model (lower is better)')
ax.set_xlabel('Model')
ax.set_ylabel('Joint NLL (TEST)')
plt.tight_layout()
plt.show()

# 2) Decompose held-out fit into choice vs RT conditional NLL components.
component_df = metric_df[['choice_only_score_test', 'rt_only_cond_score_test']]
ax = component_df.plot(
    kind='bar',
    stacked=True,
    figsize=(8, 4),
    color=['#4C78A8', '#F58518'],
)
ax.set_title('TEST NLL Decomposition: Choice + RT|Choice')
ax.set_xlabel('Model')
ax.set_ylabel('NLL contribution (TEST)')
ax.legend(['Choice NLL', 'RT|Choice NLL'])
plt.tight_layout()
plt.show()

# 3) Per-trial Joint NLL distribution (validity check for outlier sensitivity).
box_data = [
    trial_scores_long.loc[trial_scores_long['model_name'] == model_name, 'nll_joint'].to_numpy()
    for model_name in model_order
]
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(box_data, labels=model_order, showfliers=False)
ax.set_title('Per-Trial Joint NLL Distribution (TEST)')
ax.set_xlabel('Model')
ax.set_ylabel('Per-trial Joint NLL (lower is better)')
plt.tight_layout()
plt.show()

# 4) Participant-level consistency: mean per-trial Joint NLL across models.
participant_mean = trial_scores_long.groupby(['participant_id', 'model_name'], as_index=False)['nll_joint'].mean()
fig, ax = plt.subplots(figsize=(9, 4))
for participant_id, group in participant_mean.groupby('participant_id'):
    aligned = group.set_index('model_name').reindex(model_order)
    ax.plot(model_order, aligned['nll_joint'], marker='o', linewidth=1, alpha=0.35, color='gray')
mean_curve = (
    participant_mean.groupby('model_name', as_index=False)['nll_joint']
    .mean()
    .set_index('model_name')
    .reindex(model_order)['nll_joint']
)
ax.plot(model_order, mean_curve, marker='o', linewidth=2.5, color='black', label='Participant mean')
ax.set_title('Participant-Level Mean Joint NLL (TEST)')
ax.set_xlabel('Model')
ax.set_ylabel('Mean per-trial Joint NLL')
ax.legend()
plt.tight_layout()
plt.show()
